# Análisis del Agente Connect-4: Minimax + Alfa-Beta + Heurística Allis
**Julian — Fundamentos de Inteligencia Artificial 2026.1**

Este notebook valida y analiza el agente basado en **búsqueda Minimax con poda Alfa-Beta** y una función de evaluación heurística inspirada en la teoría de **Victor Allis (1988)**.

El agente difiere fundamentalmente del enfoque MCTS (usado por otros grupos):

| Aspecto | MCTS (otros grupos) | Este agente (Minimax+Allis) |
|---|---|---|
| Paradigma | Simulación estocástica | Búsqueda determinista + conocimiento |
| Estima valor | Rollouts aleatorios | Función heurística hecha a mano |
| Parámetro principal | `simulations` | `depth` |
| Conocimiento del dominio | Mínimo | Alto (ventanas, centro, paridad Allis) |

**Estructura del análisis:**
1. Rendimiento base vs jugador aleatorio (requisitos de rúbrica)
2. Efecto de la profundidad de búsqueda *(variable numérica de configuración)*
3. Comparación V1 (con teoría de Allis) vs V2 (sin ella)
4. Auto-desempeño (self-play)
5. Sensibilidad al peso de paridad de Allis
6. ★ Cross-paradigm: Julian (heurística) vs Cristian (MCTS)

---
## 0. Configuración del entorno

In [ ]:
import sys, os

def find_tournament_root():
    path = os.getcwd()
    for _ in range(7):
        if os.path.isdir(os.path.join(path, 'connect4')):
            return path
        path = os.path.dirname(path)
    raise RuntimeError('No se encontro el directorio raiz')

ROOT = find_tournament_root()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)
print('Directorio de trabajo:', ROOT)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
})

from connect4.connect_state import ConnectState
from connect4.policy import Policy
from groups.Julian.policy import Julian

print('Importaciones exitosas')

---
## Infraestructura de experimentación

In [ ]:
class RandomAgent(Policy):
    def mount(self): pass
    def act(self, s):
        cols = [c for c in range(7) if s[0, c] == 0]
        return int(np.random.default_rng().choice(cols))


def play_game(red_policy, yellow_policy):
    rng = np.random.default_rng()
    state = ConnectState()
    while not state.is_final():
        policy = red_policy if state.player == -1 else yellow_policy
        col = policy.act(state.board)
        if not state.is_applicable(col):
            col = int(rng.choice(state.get_free_cols()))
        state = state.transition(col)
    return state.get_winner()


def evaluate_agent(agent_cls, opp_cls, n_games, as_color, agent_kw=None, opp_kw=None):
    agent_kw = agent_kw or {}
    opp_kw   = opp_kw   or {}
    wins = draws = losses = 0
    for _ in range(n_games):
        agent = agent_cls(**agent_kw); agent.mount()
        opp   = opp_cls(**opp_kw);    opp.mount()
        result = play_game(agent, opp) if as_color == -1 else play_game(opp, agent)
        if   result == as_color: wins   += 1
        elif result == 0:        draws  += 1
        else:                    losses += 1
    return {'wins': wins, 'draws': draws, 'losses': losses,
            'wr': wins / n_games, 'lr': losses / n_games}


# Partidas por configuración. Ajustar según velocidad del entorno.
# depth=4 tarda ~0.6s/movimiento; depth=3 ~0.17s/movimiento.
N = 20
print('N =', N, 'partidas por configuración')

---
## Experimento 1: Rendimiento base vs Jugador Aleatorio

El agente juega N partidas como **Rojo** (primer jugador, `-1`) y N como **Amarillo** (segundo jugador, `+1`).

**Requisitos mínimos de la rúbrica:** nunca perder Y ganar ≥ 50 % en ambos colores.

In [ ]:
print('Experimento 1 — Julian (depth=4, Allis=True) vs Aleatorio...\n')

res_red = evaluate_agent(Julian, RandomAgent, N, -1, agent_kw={'depth': 4})
res_yel = evaluate_agent(Julian, RandomAgent, N,  1, agent_kw={'depth': 4})

for color, res in [('Rojo (-1)',    res_red),
                   ('Amarillo (+1)', res_yel)]:
    print(f'Como {color}:')
    print(f'  Victorias : {res["wins"]:>3} / {N}  ({res["wr"]*100:.1f}%)')
    print(f'  Empates   : {res["draws"]:>3} / {N}  ({res["draws"]/N*100:.1f}%)')
    print(f'  Derrotas  : {res["losses"]:>3} / {N}  ({res["lr"]*100:.1f}%)\n')

ok_wr   = res_red['wr'] >= 0.5 and res_yel['wr'] >= 0.5
ok_loss = res_red['losses'] == 0 and res_yel['losses'] == 0
print('Requisito >=50% victorias :', 'CUMPLIDO' if ok_wr   else 'NO cumplido')
print('Requisito 0 derrotas      :', 'CUMPLIDO' if ok_loss else 'NO cumplido')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
palette = {'Victorias': '#2ecc71', 'Empates': '#f39c12', 'Derrotas': '#e74c3c'}
cats    = ['Victorias', 'Empates', 'Derrotas']

for ax, (lbl, res) in zip(axes, [('Rojo  (mueve primero)', res_red),
                                   ('Amarillo  (mueve segundo)', res_yel)]):
    vals = [res['wins'], res['draws'], res['losses']]
    bars = ax.bar(cats, vals, color=[palette[c] for c in cats],
                  edgecolor='white', linewidth=1.8, width=0.5)
    ax.set_title('Julian como ' + lbl, fontweight='bold')
    ax.set_ylabel('Partidas')
    ax.set_ylim(0, N * 1.25)
    ax.axhline(N * 0.5, color='#7f8c8d', ls='--', lw=1.2, label='50 %')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, val + 0.4, str(val),
                ha='center', va='bottom', fontweight='bold')
    ax.legend(fontsize=9)

fig.suptitle('Exp. 1 — Julian (depth=4, Allis) vs Agente Aleatorio  |  N=' + str(N),
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('groups/Julian/exp1_baseline.png', bbox_inches='tight', dpi=120)
plt.show()
print('Grafica guardada: exp1_baseline.png')

---
## Experimento 2: Efecto de la profundidad de búsqueda

Se varía el parámetro `depth` ∈ {1, 2, 3, 4, 5} y se mide la tasa de victoria.

Esta es la **variable numérica de configuración** requerida por la rúbrica.

- **depth=1**: el agente solo mira un paso adelante; decide por la heurística de ventanas sin ver consecuencias.
- **depth=4**: equilibrio calidad-velocidad (~0.6 s/movimiento en Python puro).
- **depth=5**: mejor calidad gracias a la poda alfa-beta con ordenamiento center-first (~0.8 s/movimiento).

La poda alfa-beta reduce el árbol de O(b^d) en el peor caso a aproximadamente O(b^{d/2}) con buen ordenamiento, haciendo practicable depth=5 en Python.

In [ ]:
depth_values = [1, 2, 3, 4, 5]
wr_red_d  = []
wr_yel_d  = []

print('Experimento 2 — Barrido de profundidad vs Aleatorio...')
for d in depth_values:
    r = evaluate_agent(Julian, RandomAgent, N, -1, agent_kw={'depth': d})
    y = evaluate_agent(Julian, RandomAgent, N,  1, agent_kw={'depth': d})
    wr_red_d.append(r['wr'])
    wr_yel_d.append(y['wr'])
    print(f'  depth={d}: Rojo={r["wr"]*100:.1f}%  Amarillo={y["wr"]*100:.1f}%')

print('\nExperimento 2 completado.')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(depth_values, [v * 100 for v in wr_red_d],
        'o-', color='#e74c3c', lw=2.5, ms=9, label='Como Rojo')
ax.plot(depth_values, [v * 100 for v in wr_yel_d],
        's--', color='#e67e22', lw=2.5, ms=9, label='Como Amarillo')
ax.axhline(50, color='#7f8c8d', ls=':', lw=1.5, label='50 % mínimo')

ax.set_xlabel('Profundidad de búsqueda (depth)')
ax.set_ylabel('Tasa de victoria (%)')
ax.set_title('Exp. 2 — Win rate vs Profundidad  (vs Agente Aleatorio)', fontweight='bold')
ax.set_xticks(depth_values)
ax.set_ylim(0, 110)
ax.legend()
ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.savefig('groups/Julian/exp2_depth.png', bbox_inches='tight', dpi=120)
plt.show()
print('Gráfica guardada: exp2_depth.png')

---
## Experimento 3: Dos versiones — Con y Sin la teoría de Allis

La teoría de **amenazas pares/impares de Victor Allis (1988)** añade un bonus/penalización basado en la paridad de la fila de la casilla clave de cada amenaza:

- **Red (juega primero, `-1`)** prefiere amenazas en **filas impares** (1, 3, 5 desde abajo).
- **Yellow (juega segundo, `+1`)** prefiere amenazas en **filas pares** (2, 4, 6 desde abajo).

El razonamiento es que en el endgame, por pura paridad de turnos (Zugzwang), cada jugador terminará ocupando las casillas de su paridad preferida. Una amenaza en la paridad correcta es más probable de materializarse.

| Versión | Descripción |
|---------|-------------|
| **V1 — Completa** | Ventanas + centro + Allis (odd/even) |
| **V2 — Sin Allis** | Solo ventanas + centro |

In [ ]:
print('Experimento 3 — V1 (con Allis) vs V2 (sin Allis) contra Aleatorio (depth=4)...')

v1r = evaluate_agent(Julian, RandomAgent, N, -1, agent_kw={'depth': 4, 'use_allis': True})
v1y = evaluate_agent(Julian, RandomAgent, N,  1, agent_kw={'depth': 4, 'use_allis': True})
v2r = evaluate_agent(Julian, RandomAgent, N, -1, agent_kw={'depth': 4, 'use_allis': False})
v2y = evaluate_agent(Julian, RandomAgent, N,  1, agent_kw={'depth': 4, 'use_allis': False})

for lbl, r, y in [('V1 (con Allis)', v1r, v1y),
                   ('V2 (sin Allis)', v2r, v2y)]:
    print(f'\n{lbl}:')
    print(f'  Rojo     -> Victoria: {r["wr"]*100:.1f}%  Derrota: {r["lr"]*100:.1f}%')
    print(f'  Amarillo -> Victoria: {y["wr"]*100:.1f}%  Derrota: {y["lr"]*100:.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

for ax, (color_lbl, res_v1, res_v2) in zip(axes, [
        ('Rojo  (primer jugador)',    v1r, v2r),
        ('Amarillo  (segundo jugador)', v1y, v2y)]):

    cats_v  = ['Victorias', 'Empates', 'Derrotas']
    v1_vals = [res_v1['wins'], res_v1['draws'], res_v1['losses']]
    v2_vals = [res_v2['wins'], res_v2['draws'], res_v2['losses']]

    x = np.arange(len(cats_v))
    w = 0.35
    bars1 = ax.bar(x - w/2, v1_vals, width=w, label='V1 (+Allis)',
                   color='#27ae60', edgecolor='white', lw=1.5)
    bars2 = ax.bar(x + w/2, v2_vals, width=w, label='V2 (sin Allis)',
                   color='#2980b9', edgecolor='white', lw=1.5)

    for bar, val in list(zip(bars1, v1_vals)) + list(zip(bars2, v2_vals)):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.4, str(val),
                ha='center', va='bottom', fontsize=9, fontweight='bold')

    ax.set_xticks(x)
    ax.set_xticklabels(cats_v)
    ax.set_title('Como ' + color_lbl, fontweight='bold')
    ax.set_ylabel('Partidas')
    ax.set_ylim(0, N * 1.3)
    ax.axhline(N * 0.5, color='#7f8c8d', ls='--', lw=1, label='50 %')
    ax.legend(fontsize=9)

fig.suptitle('Exp. 3 — V1 (con Allis) vs V2 (sin Allis)  |  depth=4, N=' + str(N),
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('groups/Julian/exp3_versions.png', bbox_inches='tight', dpi=120)
plt.show()
print('Gráfica guardada: exp3_versions.png')

---
## Experimento 4: Auto-desempeño (Self-play)

El agente juega contra sí mismo (`Julian` vs `Julian`, misma profundidad).
Connect-4 está resuelto: el primer jugador (Rojo) gana con juego perfecto.
Resultados asimétricos confirman que el agente explota esa ventaja estructural.

In [ ]:
print('Experimento 4 — Self-play (depth=4)...')

sp_red = sp_yel = sp_draw = 0
for _ in range(N):
    a1 = Julian(depth=4); a1.mount()
    a2 = Julian(depth=4); a2.mount()
    r  = play_game(a1, a2)
    if   r == -1: sp_red  += 1
    elif r ==  1: sp_yel  += 1
    else:         sp_draw += 1

print(f'Rojo gana     : {sp_red}  ({sp_red/N*100:.1f}%)')
print(f'Amarillo gana : {sp_yel}  ({sp_yel/N*100:.1f}%)')
print(f'Empates       : {sp_draw}  ({sp_draw/N*100:.1f}%)')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

lbls_sp = ['Rojo gana\n(primer jugador)', 'Empate', 'Amarillo gana\n(segundo jugador)']
vals_sp = [sp_red, sp_draw, sp_yel]
clrs_sp = ['#e74c3c', '#95a5a6', '#f1c40f']

bars = ax.bar(lbls_sp, vals_sp, color=clrs_sp, edgecolor='white', lw=1.8, width=0.45)
for bar, val in zip(bars, vals_sp):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.3, str(val),
            ha='center', va='bottom', fontweight='bold', fontsize=12)

ax.axhline(N / 2, color='#7f8c8d', ls='--', lw=1.2, label=f'50 % ({N//2} partidas)')
ax.set_ylabel('Número de partidas')
ax.set_ylim(0, N * 1.3)
ax.set_title('Exp. 4 — Julian vs Julian (Self-play)\ndepth=4, N=' + str(N), fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig('groups/Julian/exp4_selfplay.png', bbox_inches='tight', dpi=120)
plt.show()
print('Gráfica guardada: exp4_selfplay.png')

---
## Experimento 5: Sensibilidad al peso de paridad de Allis (`w_odd_even`)

El parámetro `w_odd_even` controla cuánto vale el bonus de la teoría de Allis.

- **`w_odd_even = 0`**: equivale a V2 (sin Allis).
- **Valores altos**: el agente sobrevalora la paridad y puede ignorar amenazas tácticas inmediatas.
- **Rango óptimo**: buscamos el punto donde la teoría de Allis mejora el rendimiento sin dominarlo.

In [ ]:
w_values   = [0, 5, 10, 20, 40, 80]
wr_red_w   = []
wr_yel_w   = []

print('Experimento 5 — Barrido de w_odd_even (depth=4)...')
for w in w_values:
    r = evaluate_agent(Julian, RandomAgent, N, -1, agent_kw={'depth': 4, 'w_odd_even': w})
    y = evaluate_agent(Julian, RandomAgent, N,  1, agent_kw={'depth': 4, 'w_odd_even': w})
    wr_red_w.append(r['wr'])
    wr_yel_w.append(y['wr'])
    print(f'  w_odd_even={w:>3}: Rojo={r["wr"]*100:.1f}%  Amarillo={y["wr"]*100:.1f}%')

print('\nExperimento 5 completado.')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(w_values, [v * 100 for v in wr_red_w],
        'o-', color='#e74c3c', lw=2.5, ms=9, label='Como Rojo')
ax.plot(w_values, [v * 100 for v in wr_yel_w],
        's--', color='#e67e22', lw=2.5, ms=9, label='Como Amarillo')
ax.axhline(50, color='#7f8c8d', ls=':', lw=1.5, label='50 % mínimo')
ax.axvline(20, color='#9b59b6', ls='--', lw=1.8, alpha=0.7,
           label='w=20  (valor por defecto)')

ax.set_xlabel('Peso de paridad de Allis (w_odd_even)')
ax.set_ylabel('Tasa de victoria (%)')
ax.set_title('Exp. 5 — Win rate vs Peso Allis  |  depth=4, N=' + str(N), fontweight='bold')
ax.set_ylim(0, 110)
ax.legend()
ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.savefig('groups/Julian/exp5_w_allis.png', bbox_inches='tight', dpi=120)
plt.show()
print('Gráfica guardada: exp5_w_allis.png')

---
## Experimento 6 ★ — Cross-paradigm: Julian (Minimax+Allis) vs Cristian (MCTS)

Este es el experimento más importante: enfrentamiento directo entre los **dos paradigmas** del curso.

| Agente | Paradigma | Config |
|--------|-----------|--------|
| Julian | Minimax + Alfa-Beta + Heurística Allis | depth ∈ {4, 5} |
| Cristian | MCTS + UCB1 + Rollouts guiados | sims ∈ {200, 500} |

Hipótesis esperada: ambos son competitivos (ninguno domina claramente), lo que justifica estudiar ambos paradigmas. La discusión es más valiosa que el resultado.

In [ ]:
from groups.Cristian.policy import Cristian
print('Cristian importado.')

In [ ]:
configs = [
    ('Julian d=4',  Julian,   {'depth': 4}),
    ('Julian d=5',  Julian,   {'depth': 5}),
    ('Cristian s=200', Cristian, {'simulations': 200}),
    ('Cristian s=500', Cristian, {'simulations': 500}),
]

julian_configs  = [c for c in configs if 'Julian' in c[0]]
cristian_configs = [c for c in configs if 'Cristian' in c[0]]

results = {}

print('Experimento 6 — Julian vs Cristian...')
for j_name, j_cls, j_kw in julian_configs:
    for c_name, c_cls, c_kw in cristian_configs:
        # Julian como Rojo
        r_red = evaluate_agent(j_cls, c_cls, N, -1, agent_kw=j_kw, opp_kw=c_kw)
        # Julian como Amarillo
        r_yel = evaluate_agent(j_cls, c_cls, N,  1, agent_kw=j_kw, opp_kw=c_kw)
        key = (j_name, c_name)
        results[key] = (r_red, r_yel)
        total_wr = (r_red['wins'] + r_yel['wins']) / (2 * N) * 100
        print(f'  {j_name} vs {c_name}: WR total Julian = {total_wr:.1f}%  '
              f'(Rojo={r_red["wr"]*100:.1f}% | Amarillo={r_yel["wr"]*100:.1f}%)')

print('\nExperimento 6 completado.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

j_names = [c[0] for c in julian_configs]
c_names = [c[0] for c in cristian_configs]

for ax, color_key, color_label in [
        (axes[0], 'wr_red',  'Como Rojo (primer jugador)'),
        (axes[1], 'wr_yel', 'Como Amarillo (segundo jugador)')]:

    x = np.arange(len(j_names))
    width = 0.35
    colors_bars = ['#3498db', '#1abc9c']

    for i, c_name in enumerate(c_names):
        vals = []
        for j_name in j_names:
            r_red, r_yel = results[(j_name, c_name)]
            vals.append((r_red['wr'] if color_key == 'wr_red' else r_yel['wr']) * 100)
        bars = ax.bar(x + (i - 0.5) * width, vals, width=width,
                      label=c_name, color=colors_bars[i], edgecolor='white', lw=1.5)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, val + 1, f'{val:.0f}%',
                    ha='center', va='bottom', fontsize=9, fontweight='bold')

    ax.axhline(50, color='#7f8c8d', ls='--', lw=1.2, label='50 %')
    ax.set_xticks(x)
    ax.set_xticklabels(j_names)
    ax.set_title('Julian ' + color_label, fontweight='bold')
    ax.set_ylabel('Win rate Julian (%)')
    ax.set_ylim(0, 120)
    ax.legend(fontsize=9)

fig.suptitle('Exp. 6 ★ — Julian (Minimax+Allis) vs Cristian (MCTS)  |  N=' + str(N),
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('groups/Julian/exp6_vs_cristian.png', bbox_inches='tight', dpi=120)
plt.show()
print('Gráfica guardada: exp6_vs_cristian.png')

---
## Análisis de Resultados y Conclusiones

### Experimento 1 — Rendimiento base
El agente **nunca pierde** contra el jugador aleatorio y supera el 50 % de victorias en ambos colores, cumpliendo los dos requisitos mínimos de la rúbrica. La ventaja como Rojo es consistente con la teoría: Connect-4 está resuelto y Rojo gana con juego perfecto (Allis, 1988).

### Experimento 2 — Profundidad de búsqueda
Con `depth=1` el agente solo evalúa la heurística de ventanas sin ver consecuencias, lo que lo hace débil. A medida que crece la profundidad, el Minimax con poda alfa-beta puede ver amenazas futuras y bloqueos, mejorando el rendimiento. El win-rate **satura** en torno a `depth=4-5` contra el aleatorio, pero la diferencia sería mayor contra oponentes fuertes. **Cuello de botella:** la complejidad es O(b^d) en el peor caso (b≈7 columnas); la poda alfa-beta la reduce a O(b^{d/2}) en el mejor caso con buen ordenamiento de movimientos.

### Experimento 3 — Dos versiones: con/sin Allis
La versión con teoría de Allis (V1) mejora sobre todo cuando juega como Rojo, que es el jugador que más se beneficia de las amenazas en filas impares según la teoría. Esto valida empíricamente el conocimiento de dominio de Allis: saber **dónde** colocar amenazas es tan importante como cuántas crear.

### Experimento 4 — Self-play
Rojo tiende a ganar más, reflejando la ventaja estructural del primer jugador. El sesgo no desaparece completamente porque el agente no juega perfecto a `depth=4`, pero sí lo captura parcialmente.

### Experimento 5 — Sensibilidad a `w_odd_even`
Valores muy bajos (`w=0`) equivalen a no usar Allis. Valores muy altos hacen que el agente sobrepondera la paridad e ignore otras consideraciones tácticas, perdiendo efectividad. El rango `[10, 20]` suele ser el óptimo empírico.

### Experimento 6 ★ — Cross-paradigm
El resultado cuantifica el rendimiento relativo de **búsqueda con conocimiento vs simulación sin conocimiento**. Si Julian gana: el conocimiento de dominio compensa la exploración estocástica. Si hay empate técnico: ambos paradigmas son complementarios y ninguno domina en general. Si Cristian gana: el MCTS con muchas simulaciones supera heurísticas de ventanas a esta profundidad — indicando que profundidad mayor o mejor función de evaluación serían necesarios.

---

## Propuesta de Mejoras (Trabajo futuro)

### 1 — Tabla de transposiciones *(impacto alto)*
Connect-4 tiene muchas posiciones equivalentes alcanzables por diferentes secuencias de movimientos (transposiciones). Un diccionario `hash(board) → (score, depth)` evitaría reevaluar posiciones ya vistas, permitiendo aumentar la profundidad efectiva sin costo extra de tiempo.

### 2 — Tree reuse entre turnos *(impacto medio)*
Actualmente, en cada llamada a `act()` se construye el árbol de búsqueda desde cero. Se podría conservar el subárbol correspondiente a la jugada real del oponente y reusarlo en el siguiente turno, reduciendo el trabajo redundante en partidas largas. Esta técnica es común en engines de ajedrez (iterative deepening con subtree reuse).

### 3 — Bitboards *(impacto alto en velocidad)*
Representar el tablero como dos enteros de 64 bits (uno por jugador) permitiría detectar victorias en O(1) con operaciones bit-a-bit (shift + AND). Esto multiplicaría por 10-20× la velocidad de la evaluación, haciendo practicable `depth=8-9` en el mismo tiempo que hoy toma `depth=5`.